# One score over every subpopulation, and over every group

The Population Fidelity Score is a first attempt at collapsing three separate
population readings into one number: how close a model lands on each
subpopulation, how widely it spreads across them, and whether that spread runs
along the survey's own social divides. It arrived as a scalar over the whole
run. This notebook takes it apart and asks it again inside every group of
subpopulations the survey actually distinguishes.

`PFS = geometric_mean(accuracy, adaptability, structure)` over

| term | from | reads |
|---|---|---|
| `accuracy` | `1 - E`, `E = mean_i nEMD(WVS_i, LLM_i)` | 1 when the model lands on every cell |
| `adaptability` | `min(A, 1/A)`, `A = D_LLM / D_WVS` | 1 when the model spreads exactly as far as the survey; penalised symmetrically for compression and for over-spread |
| `structure` | `max(0, rho)`, Spearman over the two condensed pairwise-nEMD vectors | 1 when the model's cell-to-cell distances rank like the survey's |

A geometric mean is a bottleneck, not an average: one term at zero puts the
score at zero however good the other two are. That is the intended reading, and
`structure` is almost always the term that binds. The component plates exist so
that every zero can be traced to the term that caused it, and every row of every
table carries `binding_term`, the name of its smallest of the three.

**One thing to raise with the supervisor before this metric is used to rank
anything.** `max(0, rho)` is discontinuous at zero, and the score is a product,
so a model that correlates at `rho = -0.0001` scores exactly zero while one at
`rho = +0.0001` keeps its other two terms. That is not a hypothetical here: 30
of the 164 pooled rows land at zero, and every one of them does so on a `rho`
between `-0.026` and `-0.0001` — values no one would call negative structure,
only absent structure. The `all` view then multiplies four topics together, so
a single such topic — nearly always religion — zeroes a model's whole row, and
24 of the 41 pooled `all` rows are zero for that reason. Nothing here smooths
it: `pfs` is computed exactly as specified, the zeros are drawn as zeros, and
the ranking plate names the binding term beside every zero bar so the cliff is
visible instead of silent. If the intent was "no credit for absent structure"
rather than "no score at all", a floor on the structure term or an arithmetic
mean of the three would both keep the ordering the components plates show. The
metric-sensitivity section below scores every row with the clip removed (a
signed cube root that is continuous through zero) and under seven other
alternatives, so that trade is read off a table rather than argued.

**Center is reported beside PFS, never inside it.** `C = nEMD(mean_i WVS_i,
mean_i LLM_i)` and its score `1 - C` answer a different question — did the
model relocate its average opinion toward the target culture — and a model can
buy that cheaply while staying flat across cells. Both readings are in the
table: `pfs` is the three-term score the plates lead with, `pfs_with_center`
is the four-term form for anyone who wants the centre folded in. The plates
carry center as its own axis so the trade is visible rather than averaged away.

## Which numbers this reads

Recomputed from the run distributions, not from the derived CSVs — the same
path `population_adaptability.ipynb` takes. Two ties hold that recomputation to
the split per-run outputs `combined-metric-inputs.csv` was aggregated from:

- every per-cell `nEMD` here equals the run's own
  `outputs/**/subpopulation_distances.csv` row for row, asserted per run in the
  reconciliation cell;
- the pooled row reproduces `outputs/culture/population_adaptability.csv` —
  `e_mean_nemd`, `adaptability_ratio` and `c_center_pop_nemd` to the last digit.

One column deliberately does not match. `combined-metric-inputs.csv` spells its
`center_nEMD` from `summary_metrics.csv`, which pools *respondents*; the
`c_center_nemd` here pools *retained cells*, which is the quantity the centers
plates already draw. For Gemma-4-31B-it as released on happiness, FA, that is 0.1189
against 0.1261. Neither is wrong; they are different poolings, and this notebook
uses the cell one throughout so the centre term sits on the same footing as the
other three.

## The groups

Seven families, parsed out of the subpopulation label itself:

| family | levels |
|---|---|
| `country` | Australia, Germany, Mexico, Russia, United States |
| `wave` | 1995-1997, 2005-2006, 2017-2018 |
| `sex` | Female, Male |
| `age` | `<25` through `75+`, seven bands |
| `education` | Low, Middle, High |
| `employment` | Working, Student, Homemaker, Unemployed, Retired |
| `marital_status` | Single, Cohabiting, Married, Divorced or separated, Widowed |

From the **label**, not from the respondents inside the cell. The upstream
assignment is a matching, not a strict partition: the cell
`Mexico 2005 Female <25 High Working Single` holds 23 respondents recorded High,
11 Middle and 3 Low. The label is what defines the cell, so the label is what
defines its group. Seven cells carry an `NA` token — two on age, five on
employment — and sit out that one family while counting in every other.

Adaptability and structure are within-group quantities and are recomputed inside
each group, never averaged down from the pooled figure: a group's `A` compares
that group's own model spread against that group's own survey spread, and its
`rho` ranks only the pairs both cells of which are in the group. Accuracy and
center restrict the same way. The pooled row is therefore not the mean of its
groups and is not meant to be.

## What lands where

`figures/fidelity/` — the root holds the report over every retained cell of a
run pooled, and one subfolder per family holds the same metrics split by that
family's levels.

| plate | folder | reads |
|---|---|---|
| `fig_fidelity_ranking_<view>_<mode>` | root | PFS per run, ranked |
| `fig_fidelity_components_<view>_<mode>` | root | the three terms and the centre behind each PFS |
| `fig_fidelity_center_<view>_<mode>` | root | PFS against centre alignment, as released joined to its finetuned variant |
| `fig_fidelity_shift_<view>_<mode>` | root | change in PFS against change in centre, finetuned variants only |
| `fig_fidelity_components_home_<view>_<mode>` | root | the same terms in three blocks: the as-released conditions on all retained cells, then each finetuned variant under its as-released model on the cells of its target country |
| `fig_fidelity_shift_home_<view>_<mode>` | root | the same change read on the cells of each variant's target country |
| `fig_fidelity_cells_<view>_<mode>` | root | the per-cell nEMD spread the accuracy term averages |
| `fig_fidelity_<family>_<view>_<mode>` | `<family>/` | PFS, runs against that family's levels |
| `fig_fidelity_<family>_components_<view>_<mode>` | `<family>/` | the same four terms, one panel each |
| `fig_fidelity_<family>_shift_<view>_<mode>` | `<family>/` | change in PFS against the as-released variant |
| `fig_fidelity_subgroups_<view>` | root | PFS beside centre, one box each over the sweep's conditions, for every level of every family, NTP and FA side by side |
| `fig_fidelity_subgroups_wide_<view>_<mode>` | root | the same boxes for one mode laid along the page, levels across and score up, the shape the main text uses |
| `fig_fidelity_subgroups_models_<view>_<mode>` | root | the same rows with every model condition drawn on its own, PFS left and centre right, the sub-row carrying the variant |
| `fig_fidelity_subgroups_models_wide_<view>_<mode>` | root | the per-model plate laid along the page, PFS above centre, the sub-column carrying the variant |
| `fig_fidelity_subgroups_proprietary_<view>` | root | the proprietary lineage on the same rows: the archived GPT-3 beside the served GPT-5.6-Terra, both FA |
| `fig_fidelity_subgroups_proprietary_wide_<view>` | root | the proprietary lineage laid along the page, PFS above centre |
| `fig_fidelity_subgroups_questions_<mode>`, `fig_fidelity_subgroups_models_questions_<mode>`, `fig_fidelity_subgroups_proprietary_questions` | root | the four questions of one mode stacked in one figure, sharing the level axis and the legend, the shape the appendix uses |

| encoding | meaning |
|---|---|
| colour + marker | the model |
| fill | hollow = as released, solid = `german`, hatched = `spanish-mx` (bars cannot be half-filled) |
| bright = high | every score heatmap runs 0 to 1 on `viridis`, so bright is good — the reverse of the error heatmaps elsewhere, which run `viridis_r` |
| green / red | only on the shift heatmaps, centred on zero |
| dashed rule | a **fixed reference, never a fit**: Mixtral archived on the ranking and cell plates, zero change on the shift plates |

Five views per plate: `all`, where PFS is the geometric mean across the four
topics, then happiness, politics, religion and trust alone. Two modes. nEMD
scales with a topic's answer count, so a topic view and the `all` view are not
on the same scale and are not read against each other.

Writes `outputs/culture/population_fidelity_{cells,groups,overall,paired,paired_overall,center_correlation,center_summary,archived,sensitivity_readings,sensitivity_paired,sensitivity}.csv` and
330 plates under `figures/fidelity/`, `.png` and `.pdf` each, 660 files. Reads
`population_structure.csv` only to cross-check the pooled Pearson coefficients
of the sensitivity readings, and skips that check when the file is absent, so
it runs on its own.

A **variant** is one of the conditions a base model is evaluated under: `base`,
the model as released with nothing attached, or a culture-finetuned variant such as
`german`. The CSV column and the run-directory path segment are both still
spelled `arm`. The metric-sensitivity tables reuse the word for the nine ways
of scoring the same readings, and there it is spelled `variant`.

`.venv` has no jupyter and a bare `uv run` must not be used here — it destroys
the hand-built sm_120 `llama-cpp-python` wheel. Committed without cell outputs.

**Every finetuning effect is a paired difference on common cells.** A run keeps
only the cells with at least 20 valid answers, so a finetuned variant and its
as-released model can retain slightly different cell sets: Gemma-4-31B-it's
`german` variant keeps 630 of the 687 trust cells where its base keeps all 687.
`population_fidelity_paired.csv` therefore recomputes every term for both
conditions on the intersection of their retained cells, taken across every
replicate of both, and the `delta_*_vs_base` columns live there alone;
`population_fidelity_groups.csv` carries no deltas, each of its rows being one
run on its own cells. Where a condition was run more than once, every table
keeps one row per `replicate` and the plates draw the mean with a bar of one
standard deviation across the runs.

In [1]:
from culture.fidelity import (
    CELLS_TABLE,
    CENTER_CORRELATION_TABLE,
    CENTER_SUMMARY_TABLE,
    GROUPS_TABLE,
    OVERALL_TABLE,
    PAIRED_OVERALL_TABLE,
    PAIRED_TABLE,
    build_fidelity,
    cell_summary,
    center_correlation,
    center_correlation_summary,
    check_fidelity,
    cross_check_cells,
    overall_fidelity,
    overall_paired,
)
from culture.fidelity_baseline import manuscript_runs
from culture.fidelity_plates import fidelity_plates
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
runs = manuscript_runs(every_replicate=True)
cells, groups, paired = build_fidelity(runs=runs)
check_fidelity(cells, groups, paired)
print(len(groups), "group rows over", cells["subpopulation"].nunique(), "cells;", len(paired), "paired rows")

9548 group rows over 687 cells; 5952 paired rows


## Reconciling against the split outputs

Each run wrote its own per-cell distances at analysis time. This asserts that
what the notebook just recomputed is those same numbers, run by run, before any
of it is aggregated or drawn.

In [3]:
print(cross_check_cells(cells), "runs match their subpopulation_distances.csv")

308 runs match their subpopulation_distances.csv


## The tables

In [4]:
overall = overall_fidelity(groups)
paired_overall = overall_paired(paired)
write_csv(cell_summary(cells), CELLS_TABLE)
write_csv(groups, GROUPS_TABLE)
write_csv(overall, OVERALL_TABLE)
write_csv(paired, PAIRED_TABLE)
write_csv(paired_overall, PAIRED_OVERALL_TABLE)
print(len(overall), "rows aggregated across the four topics;", len(paired_overall), "paired")

2387 rows aggregated across the four topics; 1488 paired


## PFS against centre alignment

Do the two readings move together? For every question, mode, group and level,
`center_correlation` correlates PFS with `1 - C` across the model conditions,
Spearman and Pearson, after averaging each condition's replicates. Scope `served`
keeps the sweep's conditions, the ones the subgroup boxes span; `all` adds the two
Mixtral references, the rows the components plate draws. Fewer than three
conditions, or a score that does not vary across them, leaves the correlation blank.

Every correlation carries a 95% percentile bootstrap interval: the conditions are
resampled with replacement 10,000 times from `GLOBAL_SEED` and the correlation is
recomputed on each draw. `center_correlation_summary` sets the pooled row's
correlation beside the mean and range of the thirty per-level ones, and recomputes
that mean inside every draw: the levels share their conditions, so an interval
built from their spread as if they were independent would be far too narrow.

Writes `outputs/culture/population_fidelity_center_correlation.csv`, one row per level,
and `outputs/culture/population_fidelity_center_summary.csv`, one row per scope,
question and mode.

In [5]:
correlation = center_correlation(groups)
summary = center_correlation_summary(groups)
write_csv(correlation, CENTER_CORRELATION_TABLE)
write_csv(summary, CENTER_SUMMARY_TABLE)
print(summary.drop(columns="question_label").round(3).to_string(index=False))

 scope     question mode  n_conditions  n_levels  population_spearman  population_spearman_ci_low  population_spearman_ci_high  spearman_mean  spearman_mean_ci_low  spearman_mean_ci_high  spearman_min  spearman_max  population_pearson  population_pearson_ci_low  population_pearson_ci_high  pearson_mean  pearson_mean_ci_low  pearson_mean_ci_high  pearson_min  pearson_max
served      d_happy  ntp            18        30                0.503                      -0.013                        0.841          0.364                 0.025                  0.619        -0.085         0.678               0.395                      0.038                       0.822         0.334                0.063                 0.598       -0.110        0.720
served      d_happy   fa            19        30                0.528                       0.071                        0.815          0.433                 0.140                  0.645         0.084         0.656               0.335                    

## The archived series, scored the same way

The upstream package holds per-profile NTP probabilities for GPT-4T, Llama-3-70B and
Mixtral-8x7B and full answers for GPT-3, Llama-3-70B and Mixtral-8x7B. The tables
above carry the Mixtral pair alone. `build_archived` scores every published series
with the same `group_fidelity`, on the cells each retains, so GPT-5.6-Terra is read
against the proprietary models of the original study and not only against Mixtral.
GPT-4T was published under NTP alone and GPT-3 under FA alone, so a comparison across
them is also a comparison across elicitation modes; the Mixtral and Llama-3-70B rows,
scored under both modes on the same cells, are the bridge. The Mixtral rows reproduce
the reference rows of the groups table to the last digit, which the unit test asserts.

Writes `outputs/culture/population_fidelity_archived.csv`, shaped like the groups
table, and hands the frame to the plates so `fig_fidelity_subgroups_proprietary_<view>`
can draw the lineage: the archived GPT-3 beside the served GPT-5.6-Terra, both under FA.
GPT-4T was published under NTP alone, so it is scored in the table and kept off the
plate; the comparison stays within one mode.


In [6]:
from culture.archived_reference import ARCHIVED_TABLE, build_archived

archived = build_archived()
write_csv(archived, ARCHIVED_TABLE)
print(len(archived), "archived rows over", archived["series"].nunique(), "published series")

744 archived rows over 4 published series


## Metric sensitivity: what each implementation choice buys

The score fixes six choices a reviewer can reasonably question: the median
pairwise distance behind `A`, the symmetric `min(A, 1/A)`, Spearman's `rho`,
the clip of a nonpositive `rho` to zero, the geometric mean, and equal weight
for every cell. `build_sensitivity` walks the same runs, cells and common cells
as `build_fidelity` and takes, once per set of cells, everything a variant can
read off them — the per-cell error, both condensed pairwise vectors summarised
by median and by mean, Spearman, Pearson and Kendall over them, and the accuracy
and center terms with and without respondent weights, the weight being the
number of valid survey answers behind the cell (as low as 4 on politics; a
weight, never a retention rule). Nine variants are then pure arithmetic on those
readings:

| variant | changes |
|---|---|
| `default` | nothing: `median`, `min(A, 1/A)`, Spearman, clipped, geometric, equal cells |
| `mean_dispersion` | the mean pairwise distance in place of the median |
| `one_sided_adaptability` | `min(A, 1)`, so amplification is forgiven |
| `pearson_structure` | Pearson's `r` over the same condensed vectors |
| `kendall_structure` | Kendall's `tau_b` |
| `unclipped_structure` | `rho` kept as is, `PFS = sign(rho) * (acc * adapt * abs(rho))^(1/3)`, continuous through zero |
| `arithmetic_mean` | the arithmetic mean of the three terms, a different construct kept for the compensation it permits |
| `weighted_cells` | respondent-weighted `E` and `C`; the pair statistics stay unweighted |
| `combined` | mean dispersion, unclipped Pearson and respondent weights at once, geometric mean kept |

`sensitivity_summary` averages replicates first, then sets one row per variant
beside the default: component levels over the 164 pooled combinations as both a
mean and a median, binding counts pooled and group-resolved, nonpositive scores, rank agreement with the
default pooled and per question, NTP–FA agreement, and each finetuning arm's
paired changes pooled and inside its home country. `check_sensitivity` asserts
that the default variant reproduces `groups` and `paired` above to the last
digit, that every clipped score sits in `[0, 1]` and every signed one in
`[-1, 1]`, that the survey dispersion is the same for every complete run of a
question under both statistics, and that the pooled Pearson coefficients agree
with `population_structure.csv`, before anything is written.

Writes `outputs/culture/population_fidelity_sensitivity_readings.csv` (shaped like
the groups table), `outputs/culture/population_fidelity_sensitivity_paired.csv`
(shaped like the paired table, the base readings under `base_`) and
`outputs/culture/population_fidelity_sensitivity.csv`, one row per variant. The
manuscript's metric-sensitivity appendix is transcribed from the last one.

In [7]:
from culture.fidelity_sensitivity import (
    PAIRED_READINGS_TABLE,
    READINGS_TABLE,
    SENSITIVITY_TABLE,
    build_sensitivity,
    check_sensitivity,
    sensitivity_summary,
)

sensitivity_readings, sensitivity_paired = build_sensitivity(runs=runs)
sensitivity = sensitivity_summary(sensitivity_readings, sensitivity_paired)
check_sensitivity(sensitivity_readings, sensitivity_paired, groups, paired, sensitivity)
write_csv(sensitivity_readings, READINGS_TABLE)
write_csv(sensitivity_paired, PAIRED_READINGS_TABLE)
write_csv(sensitivity, SENSITIVITY_TABLE)
print(len(sensitivity_readings), "readings rows,", len(sensitivity_paired), "paired;", len(sensitivity), "variants")
print(sensitivity.set_index("variant").drop(columns="variant_label").round(3).T.to_string())

9548 readings rows, 5952 paired; 9 variants
variant                                     default mean_dispersion one_sided_adaptability pearson_structure kendall_structure unclipped_structure arithmetic_mean weighted_cells   combined
dispersion                                   median            mean                 median            median            median              median          median         median       mean
adaptability                              symmetric       symmetric              one_sided         symmetric         symmetric           symmetric       symmetric      symmetric  symmetric
structure                                  spearman        spearman               spearman           pearson           kendall            spearman        spearman       spearman    pearson
clipped                                        True            True                   True              True              True               False            True           True      False
aggregate  

## The plates

In [8]:
files = fidelity_plates(groups, overall, paired, paired_overall, archived=archived)
print(len(files), "files")

660 files


## Reading the plates

Start at the components plate, not the ranking. Nearly every run on these
plates scores high on accuracy, middling on adaptability and near zero on
structure, and the geometric mean is doing what it was built to do: a model
that lands close to each cell while ranking the cells wrongly is not
reproducing the population, and PFS says so. A ranking read without the
components looks like a leaderboard of models; read with them it is a
leaderboard of one term.

The centre plate is where the supervisor's question gets its answer. A run in
the bottom right has relocated its average opinion toward the German pool while
staying flat across cells — the cheap way to look adapted, and the reason
centre is kept out of the score. A run that moved up and right earned both. The
dotted connector runs from an as-released variant to its own finetuned variant, so its
slope is the trade that finetuning made.

The shift plate is the same trade differentially: right of the vertical rule is
fidelity improved, above the horizontal rule is centre improved, and each corner
names both directions; the top-left one — centre improved, fidelity worsened —
is the one to check first.

The `home` plates read the finetuning where it was aimed. The stacked component
plate keeps the as-released conditions on all retained cells in its top block
and, in the German and Mexican blocks, sets each finetuned variant directly
under its as-released model scored on that country's cells alone; GPT-5.6-Terra
and the Mixtral references, which have no variant, stay in the top block. The
`home` shift plate takes the same change on those cells, so a variant that
moved its target country's average answer without recovering the differences
among that country's cells sits in the top-left corner there too. The main text
reads both `home` plates; the population-wide pair moves to the appendix.

In the group folders, use the four topic views, not the `all` view. `all` is
the geometric mean across topics and collapses wherever any single topic scored
zero, so those heatmaps come out mostly dark and say only that the collapse
happened; the topic views next to them carry the actual comparison. Nothing is
hidden by this — it is the same knife-edge compounding one level up — but it is
the first thing to trip over when opening a group folder.

Read across a row before reading down a column. A row
that is bright everywhere except Germany is a model the German finetuning made
worse where it was fitted; a row that is bright only on Germany bought its home
cells with everyone else's. The shift heatmaps make that comparison directly,
and the components grid says which of the four terms moved.

The subgroup plate puts the two readings side by side for every level at once,
NTP left and FA right, each box spanning the sweep's conditions with the Mixtral
references left out. Read down the hollow boxes first: centre alignment barely
moves between rows, so whatever a level's PFS box does, the model's average
response was about as well placed there as anywhere. The rows whose filled box
sits far left of the pooled row while the hollow box does not — students,
cohabiting and single respondents, the youngest bands — are the groups a
centre-only reading would call well served.

The correlation table puts a number on the other direction. Across the
conditions of one level, a better-placed average goes with a somewhat higher
PFS: the per-level Spearman correlation averages between 0.26 and 0.43 over the
questions and modes, and its bootstrap interval runs from about zero to about
0.6, never near one, so centre alone does not rank the conditions by fidelity
either.

The per-model plate names what the boxes summarise: every condition on its own,
colour and marker the model, fill and sub-row the variant, both panels on the same
0 to 1 scale; the spread of a row is carried by most of the conditions, not by a
few. The proprietary plate puts the served GPT-5.6-Terra on the same rows as the
archived GPT-3, both under FA. The wide plates lay the same three readings along
the page, levels across and score up: the box plate is the shape the main text
uses, and the per-model and proprietary plates stack PFS above centre. The
`questions` plates stack the four questions of one mode in one figure, sharing
the level axis and the legend, which is how the appendix shows them.

The p-values in `rho_structure_p_value` are descriptive only — pairwise
distances over shared cells are not independent — and nothing here is a
significance claim. A group with fewer than three cells gets no `rho` and
therefore no PFS; none of the seven families has one.

Two rows carry `model_flat`: GPT-5.6-Terra as released on trust, full answers, hands Mexico
and Russia one identical distribution for every cell in the country. Every
pairwise distance is then zero, `rho` is undefined rather than low, and the
table keeps it as blank. The score does not: a model with no within-group
variation reproduces none of the group's structure by construction, so
`score_structure` is 0 there and PFS is 0 with it — which is where `A = 0`
had already put it. `model_flat` is the flag to filter on if that distinction
matters.